# M2 · Paso 1 — Verificación en Colab de `src/models.py` y `src/metrics.py`
**TFM · HAM10000** · Responsable: **Alex Reinoso** · MLflow: experimento `sandbox` (nada de aquí se cita en la memoria)

**Para qué sirve este notebook.** En local ya pasan las 22 pruebas de `tests/` con pesos aleatorios. Aquí se comprueba lo que solo se puede ver en Colab:
1. que las pruebas también pasan con Python 3.13 y las versiones de Colab;
2. que las tres redes cargan los pesos de **ImageNet** y aceptan los lotes reales de `make_dataset()` (contrato M1 → M2);
3. que `set_seeds(deterministic=True)` **no falla en la GPU** (riesgo anotado en `src/tracking.py`);
4. que compila y entrena un par de lotes en las dos fases (congelado y descongelado);
5. que los secretos y MLflow/DagsHub funcionan, con un run de prueba en `sandbox`.

No entrena nada de verdad ni guarda modelos. Gasta pocos minutos de GPU.

**Antes de empezar**
- *Entorno de ejecución → Cambiar tipo*: **T4 GPU**.
- Carpeta compartida `TFM` añadida como acceso directo en *Mi unidad* (`docs/04_colab.md` §2.1).
- Secretos activados en este notebook: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.
- La rama indicada abajo debe estar **subida a GitHub** (`git push -u origin m2/modelos-metricas`).

## 0. Parámetros

In [ ]:
OWNER  = "aReinoso007"
REPO   = "tfm-melanoma-cnn-gradcam"
BRANCH = "m2/modelos-metricas"      # rama con src/models.py y src/metrics.py
AUTOR  = "alex"

## 1. Configuración de la sesión

In [ ]:
# GPU: debe aparecer una T4. Si dice "Sin GPU", cambiar el tipo de entorno antes de seguir.
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU (entorno CPU)"
import sys; print("Python", sys.version.split()[0])

In [ ]:
# Drive: da acceso al zip del dataset y a la carpeta compartida de checkpoints
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# El repo es público: se clona sin token. Si ya estaba clonado en esta sesión, se actualiza.
import os
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
# Si falla aquí, la rama no está en GitHub: hacer push desde el portátil (ver docs/modulo2_guia_colab.md).
!git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -3

In [ ]:
# TensorFlow y Keras ya vienen en Colab; el resto sale de requirements.txt. pytest solo hace falta aquí.
!pip install -q -r requirements.txt pytest
# Sin %autoreload: en Colab (Python 3.13) falla con 'No module named imp'. Si editas un archivo de src/, reinicia la sesión.

In [ ]:
# Copiar el zip a disco local y descomprimir (leer miles de JPEG desde Drive es muy lento: ~1 min así)
import shutil, zipfile, time
from pathlib import Path
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not LOCAL.exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time()-t0:.0f} s)")

DATASET_SHA256 = "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"
SPLIT_SHA256   = "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"
n_img = len(list((LOCAL / "images").glob("*.jpg")))
assert n_img == 10015, f"[!] {n_img} imágenes, se esperaban 10015"
assert file_sha256(cfg["paths"]["metadata_csv"]) == DATASET_SHA256, "[!] Metadatos distintos a los de M1"
assert file_sha256("data/splits/split_v1.csv") == SPLIT_SHA256, "[!] El split del repo no es el v1"
print("[+] 10015 imágenes, metadatos y split v1 verificados")

## 2. Pruebas automáticas
Las mismas 22 pruebas que en local (estructura, congelado, BatchNormalization, métricas). Con las versiones de Colab deben pasar todas.

In [ ]:
!python -m pytest tests -q -p no:cacheprovider

## 3. Redes con pesos de ImageNet y lotes reales de M1
Para cada arquitectura: se construye con ImageNet, se aplica la fase 1 y la fase 2, y se pasan 2 lotes de **validación** por `make_dataset()`.
Se comprueba que la salida es `(N, 7)`, que suma 1 y que **ninguna BatchNormalization** queda entrenable.

In [ ]:
import numpy as np, yaml
from src.models import ARQUITECTURAS, build_model, congelar_base, descongelar_fraccion, bn_entrenables
from src.data import make_dataset, split_df
fase2 = cfg["training"]["phase2"]

resumen = []
for arch in ARQUITECTURAS:
    model = build_model(arch, dropout=cfg["training"]["dropout"])          # weights="imagenet"
    f1 = congelar_base(model)
    f2 = descongelar_fraccion(model, fase2["unfreeze_fraction"], fase2["freeze_batchnorm"])
    assert bn_entrenables(model) == 0
    p = model.predict(make_dataset("val", arch, training=False, cache=False).take(2), verbose=0)
    assert p.shape == (64, 7) and np.allclose(p.sum(1), 1, atol=1e-4)
    resumen.append(dict(arch=arch, fase1=f1["params_entrenables"], fase2=f2["params_entrenables"],
                        total=f2["params_total"], capas_base=f2["n_capas_base"],
                        descongeladas=f2["n_capas_descongeladas"]))
import pandas as pd
tabla = pd.DataFrame(resumen).set_index("arch")
tabla["pct_params_fase2"] = (100 * tabla.fase2 / tabla.total).round(1)
tabla

Valores esperados (medidos en local con los mismos pesos): fase 1 → 14 343 / 7 175 / 8 967 parámetros entrenables; fase 2 → 17 199 111 / 2 862 087 / 3 063 587 (ResNet50 / DenseNet121 / EfficientNetB0). Si no coinciden, parar y avisar.

## 4. Determinismo en GPU y entrenamiento mínimo
`set_seeds(..., deterministic=True)` activa `enable_op_determinism()`. Algunas operaciones de GPU no tienen versión determinista y lanzan error; es mejor descubrirlo aquí que en el run 5 de 9.
Se entrenan **3 lotes de validación** en cada fase solo para ver que compila y no falla (no se guarda nada; no es un resultado).

In [ ]:
import tensorflow as tf
from src.tracking import set_seeds
resultado_determinismo = "ok"
try:
    set_seeds(42, deterministic=True)
    arch = "efficientnetb0"          # la más ligera
    model = build_model(arch, dropout=cfg["training"]["dropout"])
    ds = make_dataset("val", arch, training=False, cache=False).take(3)
    for nombre, lr in [("fase 1", cfg["training"]["phase1"]["lr"]), ("fase 2", fase2["lr"])]:
        info = congelar_base(model) if nombre == "fase 1" else descongelar_fraccion(
            model, fase2["unfreeze_fraction"], fase2["freeze_batchnorm"])
        model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="categorical_crossentropy")  # recompilar tras cambiar trainable
        h = model.fit(ds, epochs=1, verbose=0)
        print(f"[+] {nombre}: entrenables={info['params_entrenables']:,} loss={h.history['loss'][0]:.3f}")
except Exception as e:
    resultado_determinismo = f"FALLA: {type(e).__name__}: {str(e)[:300]}"
    print("[!]", resultado_determinismo)
    print("Si es un error de determinismo: usar set_seeds(seed, deterministic=False) y registrarlo como parámetro del run.")

## 5. Métricas y tabla de predicciones sobre datos reales
Con 64 imágenes de validación (sin entrenar, así que las cifras no significan nada): solo se comprueba que las funciones corren y que la tabla tiene el formato del contrato M2 → M3.

In [ ]:
from src.metrics import calcular_metricas, prefijar, tabla_predicciones, CLASES
dfv = split_df("val").iloc[:64]
ds = make_dataset("val", "efficientnetb0", training=False, cache=False).take(2)
probs = model.predict(ds, verbose=0)
m = calcular_metricas(dfv.label.values, probs)
tp = tabla_predicciones(dfv, probs)
assert list(tp.columns) == ["image_id", "lesion_id", "y_true", "y_pred"] + [f"p_{c}" for c in CLASES]
print({k: round(v, 3) for k, v in prefijar(m, "val").items() if k in ("val_macro_f1", "val_balanced_accuracy", "val_mel_sensitivity")})
tp.head(3)

## 6. MLflow / DagsHub (run de prueba en `sandbox`)
Comprueba los secretos, la conexión y que `standard_tags()` y las métricas con prefijo se aceptan. Usa `git_dirty` para avisar si hay cambios sin commit.

In [ ]:
import math, mlflow
from google.colab import userdata
from src.tracking import setup_mlflow, standard_tags, log_config, git_dirty
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
print("MLflow:", setup_mlflow(cfg))
mlflow.set_experiment(cfg["mlflow"]["experiments"]["sandbox"])
with mlflow.start_run(run_name="m2-verificacion-paso1",
                      tags=standard_tags(cfg, modulo="m2", fase="verificacion", autor=AUTOR,
                                         determinismo_gpu=resultado_determinismo)):
    log_config(cfg)
    mlflow.log_metrics({k: v for k, v in prefijar(m, "val").items() if not math.isnan(v)})
    mlflow.log_table(tp.head(10), "predicciones_ejemplo.json")
print("[+] Run creado en el experimento sandbox. git_dirty =", git_dirty())

## 7. Resumen
Si todo salió bien: 22 pruebas en verde, la tabla del apartado 3 coincide, el apartado 4 imprime `[+]` para las dos fases y el run aparece en DagsHub → Experiments → `sandbox`.

Al terminar: *Entorno de ejecución → Desconectar y eliminar el entorno de ejecución*.